1. ensure dependencies

In [ ]:
import sys
import os
from pathlib import Path

if __name__ == "__main__":

    module_root = str(Path(os.getcwd()).parents[3])
    print(f"Module root: {module_root}")
    sys.path.insert(0, module_root)

    current_dir = str(Path(os.getcwd()).parents[2])
    print(f"root: {current_dir}")
    sys.path.insert(0, current_dir)

    if __package__ is None:
        __package__ = (
            "comfyui_image_scorer.external_modules.training_hyperparameters.text_data"
        )

    print(f"Package: {__package__}")


from ....shared.logger import configure_package_logging, get_logger, ModuleLogger

logger: ModuleLogger = get_logger(__name__)


configure_package_logging(
    level=10,
    fmt="%(message)s [%(asctime)s]",
    datefmt="%H:%M:%S",  # Full date and time
    trim_level_len=1,  # Map INFO to INF, WARNING to WRN, etc.
    trim_module_len=5,  # Only show the final name of the module
    trim_func_len=10,  # Truncate function names exceeding 12 chars
    trim_msg_len=None,  # Truncate messages exceeding 40 chars
)


2. load and process text data

In [ ]:
from ....shared.paths import text_data_file, scores_file
from ....shared.io import load_single_jsonl
from ....shared.vectors.terms import extract_terms
from tqdm import tqdm
from typing import List, Dict, Any, Tuple

# Each line in text_data.jsonl is: {"filename": {metadata_dict}}
# Each line in scores.jsonl is a raw float score (one per image, same order as index)
text_data = list(load_single_jsonl(text_data_file))
scores = list(load_single_jsonl(scores_file))
print(f"text data: {len(text_data)} lines")
print(f"scores: {len(scores)} lines")

# Validate alignment
assert len(text_data) == len(
    scores
), f"Mismatch: {len(text_data)} text entries vs {len(scores)} scores"

# Show the structure of the first entry
first_entry = text_data[0]
first_filename = next(iter(first_entry))
print(f"\nExample filename: {first_filename}")
print(f"Example data keys: {list(first_entry[first_filename].keys())}")


In [ ]:
# Aggregate data structures

from ....shared.training.plot import PlotManager

lora_data: dict[str, list[tuple[float, float]]] = {}
positive_prompt: dict[str, list[tuple[float, float]]] = {}
negative_prompt: dict[str, list[tuple[float, float]]] = {}
discrete_data: dict[str, dict[str | int, list[float]]] = {}
continuous_data: dict[str, list[tuple[float, float]]] = {}
face_age_data: dict[str, list[tuple[float, float]]] = {}
face_gender_data: dict[str, list[tuple[float, float]]] = {}
face_race_data: dict[str, list[tuple[float, float]]] = {}
pose_data: dict[str, list[tuple[float, float]]] = {}
lh_data: dict[str, list[tuple[float, float]]] = {}
rh_data: dict[str, list[tuple[float, float]]] = {}
positional_data: dict[str, dict[str, list[float]]] = {}


with tqdm(total=len(text_data), desc="Processing text data", unit="line") as pbar:
    for i in range(len(text_data)):
        outer = text_data[i]
        current_score: float = float(scores[i])
        current_line: dict[str, Any] = dict(outer[next(iter(outer))])

        PlotManager.extract_lora(current_line, current_score, lora_data)
        PlotManager.extract_prompts(
            current_line, current_score, positive_prompt, negative_prompt
        )
        PlotManager.extract_face_logits(
            current_line, current_score, face_age_data, face_gender_data, face_race_data
        )
        PlotManager.extract_face_bbox(
            current_line, current_score, continuous_data, positional_data
        )
        PlotManager.extract_pose_landmarks(
            current_line, current_score, pose_data, positional_data
        )
        PlotManager.extract_hand_landmarks(
            current_line, current_score, lh_data, rh_data, positional_data
        )
        PlotManager.extract_image_sizes(
            current_line, current_score, continuous_data, positional_data
        )
        PlotManager.extract_remaining_fields(
            current_line, current_score, discrete_data, continuous_data
        )

        pbar.update(1)

print(f"\nSummary:")
print(f"  Lora entries: {len(lora_data)} unique loras")
print(f"  Positive prompt terms: {len(positive_prompt)}")
print(f"  Negative prompt terms: {len(negative_prompt)}")
print(f"  Discrete fields: {list(discrete_data.keys())}")
print(f"  Continuous fields: {list(continuous_data.keys())}")

# --- Split continuous data into bounded (0-1) and unbounded ---
continuous_bounded, continuous_unbounded = PlotManager.split_continuous(continuous_data)
print("  Bounded continuous (0-1 range):", list(continuous_bounded.keys()))
print("  Unbounded continuous:", list(continuous_unbounded.keys()))

# --- Split positional data into bounded (0-1) and unbounded ---
pos_bounded, pos_unbounded = PlotManager.split_positional(positional_data)
print("  Positional bounded (0-1):", list(pos_bounded.keys()))
print("  Positional unbounded:", list(pos_unbounded.keys()))

# --- Split discrete data into numeric and label values ---
discrete_numeric, discrete_labels = PlotManager.split_discrete(discrete_data)
print("  Numeric discrete keys:", list(discrete_numeric.keys()))
print("  Label discrete keys:", list(discrete_labels.keys()))


3. inspect prompt terms

In [ ]:
# Show top positive prompt terms by frequency
sorted_positive = sorted(positive_prompt.items(), key=lambda x: len(x[1]), reverse=True)
print("Top 20 positive prompt terms by frequency:")
for term, data in sorted_positive[:20]:
    scores_list = [s for _, s in data]
    avg_score = sum(scores_list) / len(scores_list)
    print(f"  {term!r:40s} n={len(data):5d}  avg_score={avg_score:.4f}")


4. visualize data

In [ ]:
from ....shared.training.plot import PlotManager
%matplotlib inline

# Continuous metrics: avg score vs setting value

# Lora analysis: weight vs score scatter + aggregate bar chart
print("continuous analysis")
PlotManager.plot_continuous_analysis(lora_data, "Lora", "weight", "score")
print("aggregate summary")
PlotManager.plot_aggregate_summary(lora_data, "Lora", "score", top_percent=1, limit=10, ascending=False)


# Bounded continuous (0-1 range) - shared axes
print("Bounded continuous metrics (0-1 range, shared scales):")
print("continuous analysis")
PlotManager.plot_continuous_analysis(continuous_bounded, "Bounded metrics (0-1)", "value", "score")

# Unbounded continuous - individual scales per subplot
print("Unbounded continuous metrics (individual scales per subplot):")
print("continuous analysis")
PlotManager.plot_continuous_analysis(continuous_unbounded, "Unbounded metrics", "value", "score", cols=3, share_axes=False)

# Continuous fields scatter + summary

# Top positive prompts bar chart
print("aggregate summary")
PlotManager.plot_aggregate_summary(
    positive_prompt,
    "Positive prompts",
    "score",
    top_percent=0.01,
    limit=10,
    ascending=False,
)

# Discrete fields (sampler, scheduler, model, clip_skip, steps, etc.)
print("discrete analysis")
PlotManager.plot_discrete_analysis(discrete_numeric, "Numeric discrete", "value", "score")
print("discrete object analysis")
PlotManager.plot_discrete_object_analysis(discrete_labels)



5. positional data (face/pose/hand/size) scatter plots


In [ ]:
%matplotlib inline

print("Positional data (0-1 image coordinates) — position scatter colored by score")
PlotManager.plot_positional_data(pos_bounded, "Positional (0-1)", cols=4)

print("Bounding boxes (position + size, colored by score)")
PlotManager.plot_positional_bbox(pos_bounded, "Bounding Boxes (0-1)", cols=4)

print("Positional data (other scales) — position scatter")
PlotManager.plot_positional_data(pos_unbounded, "Positional (other)", cols=4, invert_y=False)


In [ ]:
from ....shared.training.data_transformer import list_filtered_features

print("=== Filtered Features Analysis ===")
list_filtered_features()
